[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyTorch and TensorFlow, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)

# Layers and Models


## What you will be able to do

Build a network from layers in PyTorch and in Keras, as a sequence, as a class of your own and with
Keras's functional API, and find where each framework keeps every weight and how many there are.

## The idea

### The problem

The **Gradient Descent by Hand** notebook kept a linear model's four weights and its bias in two
tensors and wrote the prediction, `X @ w + b`, out in full. The small network in the **Why PyTorch
and TensorFlow** notebook already had 193 parameters in four tensors, and a network for images can
have millions in hundreds of tensors. Written by hand, every one of those tensors needs the right
shape, its starting values, a place in the prediction, a place in the update and a way to be saved.
Change the size of one hidden layer, and three tensors change shape with it, the layer's weight and
bias and the weight of the layer after it.

### A formal definition

> A **layer** is an object that holds its own parameters and computes one step of a prediction
> from them, such as a weighted sum of its inputs plus a bias. A **model** is a layer built out of
> other layers, which can list every parameter they hold. In PyTorch both are subclasses of
> **`torch.nn.Module`**. In Keras a layer is a subclass of `keras.layers.Layer`, and a model of
> **`keras.Model`**.

### Why it works that way

When a module is assigned to an attribute of another module, as in `self.hidden = nn.Linear(4, 32)`,
PyTorch records it under that name, and `parameters()` walks every module it has recorded. That
bookkeeping is set up by `nn.Module.__init__`, so a model's own `__init__` has to call it before
assigning any layer, and a layer kept somewhere PyTorch does not look, such as an ordinary Python
list, is never found. Keras tracks the layers assigned to a model too, and it also finds layers
inside a list or a dictionary and raises nothing when one is assigned before `super().__init__()`,
so neither mistake in the Common errors happens in Keras.

The two frameworks store a linear layer's weights in opposite orientations. PyTorch's
`nn.Linear(4, 32)` keeps a weight of shape (32, 4), outputs by inputs, and computes `x @ W.T + b`.
Keras's `Dense(32)` keeps a kernel of shape (4, 32), inputs by outputs, and computes
`x @ kernel + bias`. The arithmetic is the same, and copying weights from one to the other needs a
transpose.

`nn.Linear` is given both of its sizes, so it creates its weights at once. A Keras `Dense` layer is
given only its number of outputs, and creates its weights the first time it sees an input, or when
a model starts with a `keras.Input` that says what the input will be. Until then it holds no
weights.

### Where this shows up

Every model in the rest of this guide is built this way. The **Convolutional Networks** and
**Recurrent Networks** notebooks put other kinds of layer into the same containers, and the **Saving
and Loading Models** notebook saves weights under the names this notebook prints.

### What this notebook covers

- a linear layer in each framework, and the shapes of its weights
- `nn.Sequential`, and the parameters it holds by name
- a model as a class of your own, with `__init__` and `forward`
- Keras's `Sequential`, its functional API and a subclass of `keras.Model`
- weights that Keras creates only when it first sees an input, and `summary()`
- one network's weights copied from PyTorch into Keras
- two errors, a warning and a summary with no weights, from building models

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import torch
from torch import nn

layer = nn.Linear(4, 2)
print(layer.weight.shape, layer.bias.shape)
print(sum(p.numel() for p in layer.parameters()))
```

```
torch.Size([2, 4]) torch.Size([2])
10
```

```python
import keras

dense = keras.layers.Dense(2)
dense.build((None, 4))
print(dense.kernel.shape, dense.bias.shape)
print(dense.count_params())
```

```
(4, 2) (2,)
10
```

Both layers turn four inputs into two outputs with ten parameters, eight weights and two biases.
PyTorch keeps the weights as 2 by 4 and Keras as 4 by 2.

## Setup

Two libraries and their layers, a way to quiet TensorFlow while it loads, and the stations.

- `numpy` builds the stations and checks a layer's arithmetic
- `torch` is PyTorch, and `nn` its module of layers and models
- `tensorflow` is TensorFlow, and `keras` its library of layers and models
- `warnings` catches the warning that one of the Common errors raises
- `contextlib` and `os` hide the messages TensorFlow prints while it loads

TensorFlow's C++ code prints several lines as it loads, each stamped with the time and a process
number, so no two runs would show the same output. Setting `TF_CPP_MIN_LOG_LEVEL` to 3 hides what
it logs once it has started, and `c_messages_hidden` sends the lines it prints while it starts to
`os.devnull`. The cell prints PyTorch's version without the part after `+`, which names the build.

`station_network_arrays` makes the 1,200 weather stations in Norway that the **scikit-learn, Deep
Dive** guide uses. The stations are made up, and a generator started from a fixed seed draws them,
so they are the same on every run. `scaled` holds each station's latitude, longitude, elevation and
distance from the coast, standardized with the training stations' means and standard deviations,
and `jan_mean` its January mean. The cell splits the stations into 800 for training, 200 for
validation and 200 for testing. This notebook gives the models a few stations at a time to show
what they compute, and trains one only in Your turn.

In [1]:
import contextlib
import os
import warnings

import numpy as np
import torch
from torch import nn


@contextlib.contextmanager
def c_messages_hidden():
    """Send what C and C++ code writes to the error stream to os.devnull while the block runs."""
    saved = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull, 2)
    try:
        yield
    finally:
        os.dup2(saved, 2)
        os.close(devnull)
        os.close(saved)


os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
with c_messages_hidden():
    import tensorflow as tf
    import keras


def station_network_arrays(n=1200, seed=2025):
    """The made-up stations of the scikit-learn guide, as arrays: four features and two targets."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    shift = np.where(coast_km < 3, 0.8,
            np.where(elevation > 700, -0.5,
            np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), 0.3, 0.0)))
    rng.choice(4, n, p=[0.4, 0.3, 0.1, 0.2])   # each station's owner, which this guide leaves out
    rng.integers(1950, 2025, n)                 # and the year it was installed
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    features = np.column_stack([latitude.round(2), longitude.round(2), elevation.round(0), coast_km.round(1)])
    return features.astype("float32"), jan_mean.round(1).astype("float32"), (may_mean < 2.5).astype("float32")


FEATURES = ["latitude", "longitude", "elevation_m", "coast_km"]
features, jan_mean, snow_on_may_1 = station_network_arrays()
order = np.random.default_rng(0).permutation(len(features))
train_rows, valid_rows, test_rows = order[:800], order[800:1000], order[1000:]
mean, std = features[train_rows].mean(axis=0), features[train_rows].std(axis=0)
scaled = (features - mean) / std

print("torch", torch.__version__.split("+")[0], "| tensorflow", tf.__version__, "| keras", keras.__version__,
      "|", len(features), "stations")

torch 2.11.0 | tensorflow 2.21.0 | keras 3.13.2 | 1200 stations


## Worked examples

### A linear layer in each framework

`nn.Linear(4, 32)` takes the four features and gives 32 outputs, each a weighted sum of the
features plus a bias. Its weights are drawn at random, so `torch.manual_seed` fixes them.

In [2]:
torch.manual_seed(0)
layer = nn.Linear(4, 32)
print("weight", tuple(layer.weight.shape), "| bias", tuple(layer.bias.shape))

x = torch.from_numpy(scaled[train_rows[:3]])
print("output", tuple(layer(x).shape))
print("same as x @ W.T + b:", torch.allclose(layer(x), x @ layer.weight.T + layer.bias))

weight (32, 4) | bias (32,)
output (3, 32)
same as x @ W.T + b: True


The weight is 32 by 4, outputs by inputs, and three stations in give three rows of 32 outputs.
`torch.allclose` checks that two tensors agree to within rounding, and confirms that the layer
computes `x @ W.T + b`, with its weight transposed.

In [3]:
keras.utils.set_random_seed(0)
dense = keras.layers.Dense(32, name="hidden")
print("before:", dense.built, len(dense.weights), "weights")

out = dense(scaled[train_rows[:3]])
print("after: ", dense.built, [(v.name, tuple(v.shape)) for v in dense.weights])
print("output", tuple(out.shape))
print("same as x @ kernel + bias:",
      np.allclose(out.numpy(), scaled[train_rows[:3]] @ dense.kernel.numpy() + dense.bias.numpy()))

before: False 0 weights
after:  True [('kernel', (4, 32)), ('bias', (32,))]
output (3, 32)
same as x @ kernel + bias: True


Before the call, the `Dense` layer is not built and holds no weights. Its first input has four
features, so it creates a kernel of 4 by 32, inputs by outputs, and a bias of 32. It computes
`x @ kernel + bias`, with no transpose.

### A sequence of layers

`nn.Sequential` passes its input through each layer in turn. This is the network from the **Why
PyTorch and TensorFlow** notebook, which takes four features to 32 hidden units, uses `nn.ReLU` to
replace negative values with 0, and takes the 32 units to one output. `named_parameters()` lists
every tensor the model holds, under the position of the layer it belongs to.

In [4]:
torch.manual_seed(0)
net = nn.Sequential(nn.Linear(4, 32), nn.ReLU(), nn.Linear(32, 1))
print(net)

for name, parameter in net.named_parameters():
    print(f"{name:<9} {str(tuple(parameter.shape)):<8} {parameter.numel()}")
print("total", sum(p.numel() for p in net.parameters()))

Sequential(
  (0): Linear(in_features=4, out_features=32, bias=True)
  (1): ReLU()
  (2): Linear(in_features=32, out_features=1, bias=True)
)
0.weight  (32, 4)  128
0.bias    (32,)    32
2.weight  (1, 32)  32
2.bias    (1,)     1
total 193


The `ReLU` at position 1 has no parameters, so the names jump from `0.` to `2.`. The first linear
layer holds 128 weights and 32 biases, and the second 32 weights and one bias, 193 in all, the count
the **Why PyTorch and TensorFlow** notebook printed.

### A model as a class of your own

Writing a model as a subclass of `nn.Module` gives it a name of its own, and room for anything
`nn.Sequential` cannot express, such as a layer whose input skips the one before. `__init__` creates
the layers and assigns them to attributes, after calling `super().__init__()`,
and `forward` says how an input becomes an output. The model is used by calling it, `model(x)`,
which runs `forward`.

In [5]:
class StationNet(nn.Module):
    """Four features, one hidden layer of `hidden` units, and one output."""

    def __init__(self, hidden=32):
        super().__init__()
        self.hidden = nn.Linear(4, hidden)
        self.output = nn.Linear(hidden, 1)

    def forward(self, x):
        return self.output(torch.relu(self.hidden(x)))


torch.manual_seed(0)
station_net = StationNet()
print(station_net)
print([name for name, _ in station_net.named_parameters()])
print("output", tuple(station_net(torch.from_numpy(scaled[test_rows[:5]])).shape))

StationNet(
  (hidden): Linear(in_features=4, out_features=32, bias=True)
  (output): Linear(in_features=32, out_features=1, bias=True)
)
['hidden.weight', 'hidden.bias', 'output.weight', 'output.bias']
output (5, 1)


The parameters are named after the attributes, `hidden` and `output`, where `nn.Sequential` used
positions. Five stations in give five predictions, one to a row.

### Keras's sequential, functional and subclassed models

Keras's `Sequential` is the counterpart of `nn.Sequential`. A `keras.Input` first says that each
example has four features, which lets every layer create its weights at once, and `activation="relu"`
puts the ReLU inside the `Dense` layer. Naming the layers keeps their names the same however many
layers the notebook has made before.

In [6]:
keras.utils.set_random_seed(0)
station_net_keras = keras.Sequential([
    keras.Input((4,), name="features"),
    keras.layers.Dense(32, activation="relu", name="hidden"),
    keras.layers.Dense(1, name="output"),
], name="station_net")

for weight in station_net_keras.weights:
    print(f"{weight.path:<26} {tuple(weight.shape)}")
print("total", station_net_keras.count_params())

station_net/hidden/kernel  (4, 32)
station_net/hidden/bias    (32,)
station_net/output/kernel  (32, 1)
station_net/output/bias    (1,)
total 193


Each weight's `path` names the model, the layer and the weight. The kernels are 4 by 32 and 32 by 1,
the transposes of PyTorch's weights, and the total is the same 193.

The **functional API** builds the same network by calling each layer on the output of the one
before, starting from a `keras.Input`, and then makes a `keras.Model` from the input and the final
output. It can build a model with more than one input or a branch, which `Sequential` cannot.

In [7]:
keras.utils.set_random_seed(0)
features_in = keras.Input((4,), name="features")
hidden_out = keras.layers.Dense(32, activation="relu", name="hidden")(features_in)
prediction = keras.layers.Dense(1, name="output")(hidden_out)
functional_net = keras.Model(features_in, prediction, name="station_net_functional")

print([layer.name for layer in functional_net.layers], functional_net.count_params())

['features', 'hidden', 'output'] 193


A subclass of `keras.Model` mirrors a subclass of `nn.Module`: `__init__` creates the layers and
`call` does what `forward` does. With no `keras.Input`, its layers have not seen an input, so it
holds no weights until it is first called.

In [8]:
class StationNetKeras(keras.Model):
    """Four features, one hidden layer of `hidden` units, and one output."""

    def __init__(self, hidden=32, **kwargs):
        super().__init__(**kwargs)
        self.hidden = keras.layers.Dense(hidden, activation="relu")
        self.output_layer = keras.layers.Dense(1)

    def call(self, x):
        return self.output_layer(self.hidden(x))


keras.utils.set_random_seed(0)
subclassed_net = StationNetKeras(name="station_net_subclass")
print("before the first call:", len(subclassed_net.weights), "weights")
subclassed_net(scaled[train_rows[:2]])
print("after it:", len(subclassed_net.weights), "weights,", subclassed_net.count_params(), "parameters")

before the first call: 0 weights
after it: 4 weights, 193 parameters


The model has its four weight tensors only after it has seen two stations, and then the same 193
parameters as the other two.

### What summary() shows

`summary()` prints a table of a Keras model's layers, the shape of each one's output and how many
parameters each holds. `None` in a shape stands for the batch dimension, whose size is not fixed.

In [9]:
station_net_keras.summary()

Model: "station_net"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ hidden (Dense)                  │ (None, 32)             │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 193 (772.00 B)

 Trainable params: 193 (772.00 B)

 Non-trainable params: 0 (0.00 B)

Each row gives a layer's name and type, the shape of its output, `(None, 32)` and then `(None, 1)`,
and its 160 and 33 parameters. The totals also give the parameters' size in memory, 772 bytes, which
is 193 numbers of 4 bytes each.

### A network's weights copied from PyTorch into Keras

`net` and `station_net_keras` are the same network with different random weights. Copying `net`'s
weights into the Keras model with `set_weights`, each weight matrix transposed with `.T`, makes them
the same model. `detach()` gives a tensor with no history, which `.numpy()` needs.

In [10]:
first, second = net[0], net[2]
station_net_keras.get_layer("hidden").set_weights(
    [first.weight.detach().numpy().T, first.bias.detach().numpy()])
station_net_keras.get_layer("output").set_weights(
    [second.weight.detach().numpy().T, second.bias.detach().numpy()])

stations = scaled[test_rows[:3]]
with torch.no_grad():
    print("PyTorch:", net(torch.from_numpy(stations)).numpy().ravel().round(4))
print("Keras:  ", station_net_keras.predict(stations, verbose=0).ravel().round(4))

PyTorch: [-0.2096  0.2303 -0.3354]
Keras:   [-0.2096  0.2303 -0.3354]


In [11]:
try:
    station_net_keras.get_layer("hidden").set_weights(
        [first.weight.detach().numpy(), first.bias.detach().numpy()])
except ValueError as error:
    print(f"ValueError: {error}")

ValueError: Layer hidden weight shape (4, 32) is not compatible with provided weight shape (32, 4).


The two models give the same three predictions to four decimal places. Without `.T`, `set_weights`
raises a `ValueError` at the first layer, whose kernel is 4 by 32 where PyTorch's weight is 32 by 4.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/06-layers-and-models-solutions.ipynb).

**1.** Build an `nn.Sequential` with two hidden layers of 16 units, each followed by `nn.ReLU()`, and one output, then print it and its number of parameters.

In [12]:
# your code here

**2.** Write a subclass of `nn.Module`, `TwoLayerNet`, whose `__init__` takes the sizes of two hidden layers, and print the name and shape of every parameter for hidden sizes of 8 and 4.

In [13]:
# your code here

**3.** Build the network from task 1 with Keras's functional API, with named layers, and check that `count_params()` matches the PyTorch count.

In [14]:
# your code here

**4.** Copy the weights of the PyTorch network from task 1, made from `torch.manual_seed(0)`, into the Keras network from task 3, and print both models' predictions for the first three test stations.

In [15]:
# your code here

**5.** Train a `StationNet` made from `torch.manual_seed(0)` with `torch.optim.Adam(model.parameters(), lr=0.01)` and `nn.MSELoss()` for 500 full-batch steps on the training stations, and print its loss on the test stations.

In [16]:
# your code here

## Common errors

### `AttributeError: cannot assign module before Module.__init__() call`

In [17]:
class ForgotInit(nn.Module):
    def __init__(self):
        self.hidden = nn.Linear(4, 32)


ForgotInit()

AttributeError: cannot assign module before Module.__init__() call

`nn.Module.__init__` creates the record in which PyTorch keeps each layer assigned to the model, so
assigning a layer before it has run raises this error. Call `super().__init__()` as the first
line of `__init__`, as `StationNet` does.

### `ValueError: optimizer got an empty parameter list`

In [18]:
class ListNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = [nn.Linear(4, 32), nn.Linear(32, 1)]

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


list_net = ListNet()
print(list_net, "|", len(list(list_net.parameters())), "parameters")
torch.optim.SGD(list_net.parameters(), lr=0.1)

ListNet() | 0 parameters


ValueError: optimizer got an empty parameter list

PyTorch looks for layers among a module's attributes, and a Python list is not a module, so the two
layers inside it are invisible. The model prints as `ListNet()`, with nothing inside, and has no
parameters to hand to an optimizer. `nn.ModuleList` holds layers in a list that PyTorch does look
inside.

In [19]:
class ModuleListNet(ListNet):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(4, 32), nn.Linear(32, 1)])


module_list_net = ModuleListNet()
print(len(list(module_list_net.parameters())), "parameters")
optimizer = torch.optim.SGD(module_list_net.parameters(), lr=0.1)

4 parameters


With `nn.ModuleList`, the same two layers give four parameter tensors, and the optimizer accepts
them.

### ``UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer.``

Older Keras code tells the first layer its input size with `input_shape`.

In [20]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", UserWarning)
    old_style = keras.Sequential([
        keras.layers.Dense(32, activation="relu", input_shape=(4,)),
        keras.layers.Dense(1),
    ])
for warning in caught:
    print(f"{warning.category.__name__}: {warning.message}")

Keras makes the model and warns, and the rest of the message names the form it expects. Put
`keras.Input((4,))` at the start of the list in place of `input_shape=(4,)`, as the worked examples
do.

### No error, and a summary of a model with no weights

In [21]:
no_input = keras.Sequential([
    keras.layers.Dense(32, activation="relu", name="hidden"),
    keras.layers.Dense(1, name="output"),
], name="no_input")
print(len(no_input.weights), "weights")
no_input.summary()

0 weights


Model: "no_input"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ hidden (Dense)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

With no `keras.Input`, the model cannot know how many features its first layer will receive, so
neither layer has created its weights. The summary shows `?` for every output shape and
`0 (unbuilt)` for every count, and raises nothing. Give the model a `keras.Input`, or call `build`
with the shape of its input.

In [22]:
no_input.build((None, 4))
print(len(no_input.weights), "weights,", no_input.count_params(), "parameters")

4 weights, 193 parameters


Built for four features, the same layers hold four weight tensors and 193 parameters.

## Recap

- You can build a network as `nn.Sequential` or as a subclass of `nn.Module`, and as a Keras
  `Sequential`, a functional model or a subclass of `keras.Model`.
- You can list a model's parameters by name, count them and hand them to an optimizer.
- You can read the shape of a linear layer's weights in each framework, and transpose them to copy a
  network from one to the other.
- You can tell when a Keras model has created its weights, and give it a `keras.Input` so that it
  does so at once.

## What is next

The **Reading Shape Errors** notebook gives these layers inputs of the wrong shape, and reads what
each number in the messages they raise refers to.

---

&#8592; **Previous:** [Gradient Descent by Hand](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/05-gradient-descent-by-hand.ipynb)  &nbsp;·&nbsp;  [PyTorch and TensorFlow, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Reading Shape Errors](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/07-reading-shape-errors.ipynb) &#8594;
